# NB-02 — CLIP Encoder & Contrastive Learning

**Goal:** Understand how CLIP aligns images and text in a shared embedding space.

**Key questions:**
1. How does CLIP enable zero-shot classification without task-specific training?
2. What is contrastive learning and InfoNCE loss?
3. How do we use CLIP for retrieval and similarity search?

---

## Concept: Contrastive Learning in Plain English

CLIP trains two encoders (image + text) so that **matching pairs are close** and **non-matching pairs are far apart** in embedding space.

```
Training batch of N (image, caption) pairs:

  Image Encoder → [img_1, img_2, ..., img_N]
  Text Encoder  → [txt_1, txt_2, ..., txt_N]

  Similarity matrix (N×N):
              txt_1  txt_2  txt_3
    img_1  [  HIGH    low    low  ]  ← img_1 should match txt_1
    img_2  [  low    HIGH    low  ]
    img_3  [  low    low    HIGH  ]

  InfoNCE loss: maximize diagonal, minimize off-diagonal
```

At inference, we never fine-tune — we just compare a new image to text labels and pick the highest score (**zero-shot classification**).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import BytesIO
from PIL import Image

from src.encoders.clip_encoder import CLIPVisionEncoder

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

def load_image(url: str, fallback_color: tuple[int, int, int]) -> Image.Image:
    try:
        resp = requests.get(url, timeout=10)
        return Image.open(BytesIO(resp.content)).convert("RGB")
    except Exception:
        arr = np.full((224, 224, 3), fallback_color, dtype=np.uint8)
        return Image.fromarray(arr)

print("Loading CLIP (ViT-L/14)... first run downloads ~1.7GB")
encoder = CLIPVisionEncoder(device=DEVICE)
print(f"Hidden size: {encoder.hidden_size}, patches: {encoder.num_patches}")

## 1. Zero-Shot Classification

Give CLIP 5 images and 5 text labels — no training. Build a similarity matrix and see if the diagonal wins.

In [ ]:
IMAGE_URLS = [
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg", (180, 120, 80)),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg", (200, 180, 160)),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/320px-PNG_transparency_demonstration_1.png", (80, 160, 220)),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/e/e9/Felis_silvestris_silvestris_small_gradual_decrease_of_quality.png/320px-Felis_silvestris_silvestris_small_gradual_decrease_of_quality.png", (160, 140, 100)),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg", (210, 190, 120)),
]

LABELS = ["a photo of a dog", "a photo of a cat", "a colorful abstract image", "a wild cat in nature", "a yellow labrador retriever"]
CATEGORIES = ["dog", "cat", "abstract", "wildcat", "labrador"]

images = [load_image(url, color) for url, color in IMAGE_URLS]

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, img, cat in zip(axes, images, CATEGORIES):
    ax.imshow(img)
    ax.set_title(cat)
    ax.axis("off")
plt.suptitle("5 Test Images")
plt.tight_layout()
plt.show()

sim_matrix = encoder.compute_similarity(images, LABELS).cpu().numpy()
print("Similarity matrix shape:", sim_matrix.shape)
print(sim_matrix.round(3))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    sim_matrix,
    annot=True,
    fmt=".2f",
    xticklabels=[c[:10] for c in CATEGORIES],
    yticklabels=CATEGORIES,
    cmap="viridis",
)
plt.title("CLIP Zero-Shot Similarity Matrix")
plt.xlabel("Text label")
plt.ylabel("Image")
plt.tight_layout()
plt.show()

predictions = sim_matrix.argmax(axis=1)
for i, pred_idx in enumerate(predictions):
    print(f"Image '{CATEGORIES[i]}' → predicted '{CATEGORIES[pred_idx]}' (score={sim_matrix[i, pred_idx]:.3f})")

## 2. Image vs Description — Same Concept?

Compare an image embedding against a literal description vs an unrelated caption.

In [ ]:
dog_image = images[0]
queries = [
    "a photo of a dog",
    "a fluffy canine sitting outdoors",
    "a photo of an airplane",
    "a bowl of soup",
]

scores = encoder.compute_similarity([dog_image], queries)[0].cpu().numpy()
for q, s in zip(queries, scores):
    bar = "█" * int(max(0, s) * 30)
    print(f"{s:+.3f}  {bar:<30}  {q}")

## 3. UMAP Embedding Space Visualization

Embed 20 images (4 per category), project to 2D with UMAP, and color by category.

In [ ]:
try:
    import umap
except ImportError:
    raise ImportError("Install umap-learn: pip install umap-learn")

from transformers import CLIPModel, CLIPProcessor

# Build a larger image set by repeating with slight color shifts
base_images = images[:4]
gallery: list[Image.Image] = []
gallery_labels: list[str] = []

for label, img in zip(CATEGORIES[:4], base_images):
    for shift in range(5):
        arr = np.array(img).astype(np.int16)
        arr = np.clip(arr + shift * 8, 0, 255).astype(np.uint8)
        gallery.append(Image.fromarray(arr))
        gallery_labels.append(label)

processor = CLIPProcessor.from_pretrained(encoder.model_id)
clip_model = CLIPModel.from_pretrained(encoder.model_id).to(DEVICE).eval()

with torch.no_grad():
    inputs = processor(images=gallery, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)
    embeddings = clip_model.get_image_features(pixel_values=pixel_values)
    embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)

reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(embeddings.cpu().numpy())

plt.figure(figsize=(8, 6))
unique_labels = sorted(set(gallery_labels))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
for label, color in zip(unique_labels, colors):
    mask = [l == label for l in gallery_labels]
    pts = coords[mask]
    plt.scatter(pts[:, 0], pts[:, 1], label=label, alpha=0.8, s=60, c=[color])
plt.legend()
plt.title("UMAP of CLIP Image Embeddings (20 images)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()

## 4. Text-Guided Image Retrieval

Given a text query, find the closest image from a gallery.

In [ ]:
gallery = images
gallery_names = CATEGORIES
query = "a golden retriever dog"

scores = encoder.compute_similarity(gallery, [query]).squeeze(1).cpu().numpy()
ranked = sorted(zip(gallery_names, scores, gallery), key=lambda x: x[1], reverse=True)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, score, img) in zip(axes, ranked[:3]):
    ax.imshow(img)
    ax.set_title(f"{name}\nscore={score:.3f}")
    ax.axis("off")
plt.suptitle(f"Top-3 retrieval for: '{query}'")
plt.tight_layout()
plt.show()

## 5. Sequence Features for the Multimodal Pipeline

For our QA system, we need **token sequences** (not just pooled vectors) to feed the projector.

In [ ]:
seq_feats = encoder.encode_image([images[0]])
print(f"encode_image() shape: {seq_feats.shape}")
print(f"  → [batch, num_patches+1, hidden_size]")
print(f"  → CLIP ViT-L/14: [1, 257, 1024]")
print()
print("We'll pass these patch tokens to the MLP/Q-Former projector in Phase 2.")

## ✅ Phase 1 (Part B) Checklist

- [ ] I understand contrastive learning and InfoNCE
- [ ] I can explain zero-shot classification with CLIP
- [ ] I know CLIP outputs `[batch, seq_len, hidden]` for the vision tower
- [ ] `CLIPVisionEncoder` in `src/encoders/clip_encoder.py` is tested

**Next:** Phase 2 — Input Projectors (`NB-03-linear-projector.ipynb`)